In [0]:
from pyspark.sql import functions as f
from functools import reduce

# Load Data from Bronze and Silver Layers

This step loads the data from the Bronze and Silver tables into separate Spark DataFrames.

The Bronze data will be used as a reference for data validation, including record count consistency checks.

The Silver data will be used for data validation and as the source for all subsequent transformations required to build the dimension and fact tables in the Gold layer.

In [0]:
# Define source silver and bronze table
SILVER_TABLE = "workspace.weather_data.weather_silver"
BRONZE_TABLE = "workspace.weather_data.weather_raw"
# Load transformed data from bronze and silver tables
silver_df = spark.read.table(SILVER_TABLE)
bronze_df = spark.read.table(BRONZE_TABLE)

# Display schemas and few records from both tables
print("Bronze table schema:")
bronze_df.printSchema()
print("Silver table schema:")
silver_df.printSchema()

print("\nBronze table records:")
display(bronze_df.limit(10))
print("\nSilver table records:")
display(silver_df.limit(10))

%md
# Validate Record Count Consistency Between Bronze and Silver Layers

This step compares the total number of records stored in the Bronze and Silver layers.

The Silver layer applies data transformations without adding or removing records.

The record counts must match between both layers.

The pipeline stops if a difference in record counts is detected.

In [0]:
# Counting rows in Bronze and Silver Tables
rows_bronze = bronze_df.count()
rows_silver = silver_df.count()
# Check if the number of rows in the silver table is equal to the number of rows in the bronze table
if rows_bronze != rows_silver:
    raise ValueError(
        f"Data quality check failed: Record count in bronze table: {rows_bronze}; Record count in silver table: {rows_silver}"
    )
else:
    print(f"Data quality check passed: There are {rows_bronze} rows in both tables")

# Create the City Dimension Table

Starting from this step, SQL is used to perform data transformations and create dimension and fact tables in the Gold layer.

This approach separates data ingestion and preparation, handled with PySpark, from analytical data modeling, implemented using SQL.

The `dim_city` dimension table is created using data from the Silver layer.

The transformation consists of the following operations:

- Extract unique combinations of city names and geographic coordinates.
- Generate a unique surrogate key (`city_id`) for each city using the `ROW_NUMBER()` window function.
- Store the resulting dataset as a Delta table.

The `dim_city` table contains one record per city and serves as a reference dimension for the `fact_weather` table.

The table is created using `CREATE OR REPLACE TABLE`, allowing the transformation to be rerun and the dimension table to be fully refreshed.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.weather_data.dim_city
  USING DELTA AS
SELECT
  -- Adding a surrogate key to the dim_city table
  ROW_NUMBER() OVER (ORDER BY city) AS city_id,
  *
FROM
  (
    -- Using subquery to select distinct city, latitude, longitude
    SELECT
    DISTINCT
      city,
      latitude,
      longitude
    FROM
      workspace.weather_data.weather_silver
  )

# Create the Date Dimension Table

This step creates the `dim_date` dimension table in the Gold layer using a dynamically generated calendar.

The calendar is generated independently of the individual records in the Silver layer. The Silver table is used only to determine the latest available date.

The transformation consists of the following operations:

- Define a fixed calendar start date of January 1, 2020.
- Retrieve the maximum available date from the Silver layer to determine the calendar end date.
- Generate a continuous sequence of calendar dates using the `SEQUENCE` and `EXPLODE` functions.
- Generate a surrogate key (`date_key`) for each date using the `yyyyMMdd` format.
- Derive calendar attributes, including day, month, month name, quarter, year, day of the week, and weekend indicator.
- Store the resulting dataset as a Delta table.

The `dim_date` table contains one record per calendar day, including dates without corresponding weather measurements.

The dynamic end date supports future incremental loading by allowing the calendar to expand as new dates become available in the Silver layer.

The initial table is created using `CREATE OR REPLACE TABLE`. Incremental loading will be implemented as a separate process in a subsequent development step.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.weather_data.dim_date
  USING DELTA AS
-- Create a table with all dates between 2020-01-01 and the most recent date in the silver table
WITH dates AS (
  SELECT
    EXPLODE(SEQUENCE(DATE "2020-01-01", MAX(DATE), INTERVAL 1 DAY)) AS full_date
  FROM
    workspace.weather_data.weather_silver
)
--Adding additional columns for date_key, day, month, quarter, year, day_of_week, is_weekend
--Note: 0 = Sunday, 1 = Monday
SELECT
  CAST(DATE_FORMAT(full_date, "yyyyMMdd") AS INT) AS date_key,
  full_date,
  DAY(full_date) AS day,
  MONTH(full_date) AS month,
  MONTHNAME(full_date) AS month_name,
  QUARTER(full_date) AS quarter,
  YEAR(full_date) AS year,
  WEEKDAY(full_date) + 1 AS day_of_week,
  CASE
    WHEN (weekday(full_date) + 1) IN (6, 7) THEN TRUE
    ELSE FALSE
  END AS is_weekend
FROM
  dates

# Create the Weather Fact Table

This step creates the `fact_weather` fact table in the Gold layer using data from the Silver layer and the dimension tables.

The transformation consists of the following operations:

- Join the Silver table with `dim_city` using the city name.
- Join the Silver table with `dim_date` using the `date` column from Silver and the `full_date` column from `dim_date`.
- Retrieve the corresponding surrogate keys (`city_id` and `date_key`).
- Select the hourly weather measurements required for analytical reporting.
- Store the resulting dataset as a Delta table.

The `fact_weather` table uses an hourly grain, with one record per city and measurement timestamp.

The table contains foreign keys referencing `dim_city` and `dim_date`, the original measurement timestamp, and weather measurements.

The table is created using `CREATE OR REPLACE TABLE`, allowing the transformation to be rerun and the fact table to be fully refreshed.

In [0]:
%sql
--Create Fact Table
CREATE OR REPLACE TABLE workspace.weather_data.fact_weather
USING DELTA
SELECT
  dc.city_id,
  dd.date_key,
  ws.measurement_time,
  ws.temperature_2m,
  ws.apparent_temperature,
  ws.relative_humidity_2m,
  ws.precipitation,
  ws.rain,
  ws.snowfall,
  ws.cloud_cover,
  ws.wind_speed_10m,
  ws.wind_gusts_10m,
  ws.wind_direction_10m,
  ws.shortwave_radiation
FROM
  workspace.weather_data.weather_silver ws
  --Joining silver table with dim tables to aquire surrogate keys
    LEFT JOIN workspace.weather_data.dim_city dc
      ON dc.city = ws.city
    LEFT JOIN workspace.weather_data.dim_date dd
      ON dd.full_date = ws.date

# Validate Record Count Consistency Between Silver and Gold Layers

This step compares the total number of records stored in the Silver layer with the total number of records in the Gold `fact_weather` table.

The `fact_weather` table preserves the hourly data grain from the Silver layer without aggregating or removing records.

The record counts must match between both layers.

The pipeline stops if a difference in record counts is detected.

In [0]:
# Establishing dataframe from fact_weather table
fact_weather_df = spark.read.table("workspace.weather_data.fact_weather")

# Counting rows in fact_weather and weather_silver tables
fact_weather_count = fact_weather_df.count()
silver_count = silver_df.count()

# Check if the number of rows in the silver table is equal to the number of rows in the fact_weather table
if fact_weather_count == silver_count:
    print(f"Data quality check passed: There are {silver_count} rows in both tables")
else:
    raise ValueError(
        f"Data quality check failed: Record count in silver table: {silver_count}; Record count in fact_weather table: {fact_weather_count}"
    )

# Define a Reusable Key Uniqueness Validation Function

This step defines a reusable function to validate the uniqueness of key columns across the Gold layer tables.

The function accepts the following arguments:

- `dataframe` – Spark DataFrame containing the data to validate.
- `table` – name of the table being validated, used in validation messages.
- `*key_column` – one or more column names containing the key values to check for duplicates.

The function supports both single-column keys and composite keys.

The validation consists of the following operations:

- Count all records in the specified table.
- Count distinct key values or distinct combinations of key values across all specified columns.
- Compare both counts to identify duplicate keys.
- Raise a `ValueError` if duplicate keys are detected.

For single-column keys, the function checks the uniqueness of individual key values.

For composite keys, the function checks the uniqueness of combinations of values across all specified key columns.

The function can be reused to validate surrogate keys in dimension tables and composite keys in fact tables.

This approach reduces code duplication and ensures consistent key uniqueness validation across the Gold layer tables.

In [0]:
def key_uniqueness(dataframe, table, *key_column):
    # Count rows in dataframe
    dataframe_count = dataframe.count()
    # Count unique values in tables key column
    unique_dataframe_key_count = dataframe.select(*key_column).distinct().count()

    # Check if number of rows in dataframe is equal to number of unique values in tables key column
    if dataframe_count == unique_dataframe_key_count:
        print(f"Data quality check passed: Every row in {table} table has a unique key")
    else:
        raise ValueError(
            f"Data quality check failed: There are {dataframe_count} rows in {table} table, but only {unique_dataframe_key_count} keys"
        )

# Validate Dimension Key Uniqueness

This step verifies that the surrogate keys in the dimension tables are unique.

The validation consists of the following checks:

- Compare the total number of records in `dim_date` with the number of distinct `date_key` values.
- Compare the total number of records in `dim_city` with the number of distinct `city_id` values.

Each dimension key must uniquely identify one record in its respective dimension table.

The pipeline stops if duplicate dimension keys are detected.

In [0]:
# Check Uniqueness of the date_key column in dim_date table
# Establish dataframe from dim_date table
dim_date = spark.read.table("workspace.weather_data.dim_date")

key_uniqueness(dim_date, "dim_date", "date_key")

# Check Uniqueness of the city_id column in dim_city table
# Establish dataframe from dim_city table
dim_city = spark.read.table("workspace.weather_data.dim_city")

key_uniqueness(dim_city, "dim_city", "city_id")

# Validate Fact Table Grain

This step verifies that the `fact_weather` table maintains the expected hourly data grain.

The expected grain is one record per city and measurement timestamp.

The validation consists of the following operations:

- Count all records in the `fact_weather` table.
- Count distinct combinations of `city_id` and `measurement_time`.
- Compare both counts to identify duplicate records at the expected grain.

Each combination of `city_id` and `measurement_time` must be unique.

The pipeline stops if duplicate records are detected.

In [0]:
# Count unique composite keys in fact_weather table

# Declare a list with key_columns in fact_weather
fact_weather_composite_key = ["city_id", "measurement_time"]

# Check uniqueness of the composite key in fact_weather table
key_uniqueness(fact_weather_df, "fact_weather", *fact_weather_composite_key)

# Define a Reusable Referential Integrity Validation Function

This step defines a reusable function to validate referential integrity between primary and foreign key columns across two DataFrames.

The function accepts the following arguments:

- `primary_df` – DataFrame containing the primary key columns.
- `foreign_df` – DataFrame containing the foreign key columns.
- `primary_table` – name of the primary table, used in validation messages.
- `foreign_table` – name of the foreign table, used in validation messages.
- `primary_keys` – list of column names representing the primary key.
- `foreign_keys` – list of column names representing the foreign key.

The function supports both single-column keys and composite keys.

## Validation Process

The validation consists of the following operations:

- Check whether the primary and foreign key lists contain the same number of columns.
- Select distinct foreign key combinations and primary key combinations.
- Pair corresponding primary and foreign key columns using the `zip()` function.
- Build a single JOIN condition by combining all column comparisons using `reduce()` and the logical AND operator (`&`).
- Use a left anti join to identify foreign key combinations that do not have matching records in the primary table.
- Raise a `ValueError` if any foreign key combinations are missing from the primary table.

## Requirements for Correct Function Execution

The function requires the following conditions to be met:

1. **Equal number of key columns:** The `primary_keys` and `foreign_keys` lists must contain exactly the same number of elements.

2. **Matching column positions:** Corresponding columns must be placed at the same indices in both lists.

3. **Matching business meaning:** Columns at corresponding indices must represent the same business attributes, even if their names differ.

Failure to meet these requirements may result in incorrect JOIN conditions or validation results.

## Reusability

The function can be reused to validate referential integrity between dimension and fact tables, including relationships based on composite keys.

This approach reduces code duplication and ensures consistent referential integrity validation across the Gold layer.

In [0]:
def referential_integrity_check(
    primary_df, foreign_df, primary_table, foreign_table, primary_keys, foreign_keys
):
    # Check if primary_keys and foreign_keys are passed in as lists
    if not (isinstance(primary_keys, list)) or not (isinstance(foreign_keys, list)):
        raise TypeError("primary_keys and foreign_keys must be passed in as lists")

    # Both key lists must have the same length
    if len(primary_keys) != len(foreign_keys):
        raise ValueError(
            f"There are {len(primary_keys)} keys in primary_keys argument and {len(foreign_keys)} keys in foreign_keys argument. Primary and foreign keys must contain the same number of columns"
        )

    # Key lists must have at least one value in each list
    if not primary_keys or not foreign_keys:
        raise ValueError(
            "There are no values in primary_keys and foreig_keys variables"
        )

    # Select only unique keys from both dataframes
    unique_foreign_keys_df = foreign_df.select(*foreign_keys).distinct()
    primary_key_df = primary_df.select(*primary_keys).distinct()

    # Declare JOIN condition for any number of keys
    join_condition = reduce(
        lambda condition, pair: condition & pair,
        [
            f.col(f"primary.{primary_key}") == f.col(f"foreign.{foreign_key}")
            for primary_key, foreign_key in zip(primary_keys, foreign_keys)
        ],
    )
    print(join_condition)

    # Create a dataframe from unique_foreign_keys_df that only has keys that are not in the primary_key_df
    lacking_foreign_keys_df = unique_foreign_keys_df.alias("foreign").join(
        primary_key_df.alias("primary"), on=join_condition, how="left_anti"
    )

    # If lacking_foreign_keys_df has no keys then pass otherwise fail
    if lacking_foreign_keys_df.count() == 0:
        print(
            f"Data quality check passed: All keys in {foreign_table} are present in {primary_table}"
        )
    else:
        raise ValueError(
            f"Data quality check failed: Top 10 foreign_keys that are missing from {primary_table} table: {lacking_foreign_keys_df.limit(10).collect()}"
        )

# Validate Referential Integrity Between Fact and Dimension Tables

This step verifies that all foreign keys in the `fact_weather` table reference existing records in the corresponding dimension tables.

The validation consists of the following checks:

- Verify that every `city_id` in `fact_weather` exists in `dim_city`.
- Verify that every `date_key` in `fact_weather` exists in `dim_date`.

Referential integrity ensures that every weather measurement is correctly associated with an existing city and calendar date.

The pipeline stops if any foreign key references a missing dimension record.

In [0]:
# Referential integrity check for dim_city
referential_integrity_check(primary_df = dim_city, foreign_df = fact_weather_df, primary_table = "dim_city", foreign_table = "fact_weather", primary_keys = ["city_id"], foreign_keys = ["city_id"])

# Referential integrity check for dim_date
referential_integrity_check(primary_df= dim_date, foreign_df = fact_weather_df, primary_table = "dim_date", foreign_table = "fact_weather", primary_keys =["date_key"], foreign_keys = ["date_key"])

# Define a Reusable NULL Validation Function

This step defines a reusable function to validate NULL values in critical columns across the Gold layer tables.

The function accepts the following arguments:

- `dataframe` – Spark DataFrame containing the data to validate.
- `table_name` – name of the table being validated, used in validation messages.
- `*columns` – variable number of column names to check for NULL values.

The validation consists of the following operations:

- Count NULL values in all specified columns using a single Spark aggregation.
- Display the validation result for each column.
- Calculate the total number of NULL values across all specified columns.
- Raise a `ValueError` if any NULL values are detected.

The function is reused to validate critical columns in `fact_weather`, `dim_city`, and `dim_date`.

This approach reduces code duplication and ensures consistent NULL validation across the Gold layer tables.

In [0]:
def tables_null_validation(dataframe, table_name, *columns):
    # Declare the table that s being checked
    print(f"Checking null values in critical columns in {table_name} table")

    # Create a dataframe with the count of null values for each column
    null_count_df = dataframe.agg(
        *[
            (f.sum(f.when(f.col(column).isNull(), 1).otherwise(0))).alias(column)
            for column in columns
        ]
    )

    # Extract the counts as a list
    null_counts = null_count_df.collect()[0]
    total_nulls = sum(null_counts)

    # Check for null values in each column
    for column, count in zip(columns, null_counts):
        if count == 0:
            print(f"Data quality check passed: no null values found in {column} column")
        else:
            print(
                f"Data quality check failed: {count} null values found in {column} column"
            )

    # Stop the pipeline if null values are found
    if total_nulls != 0:
        raise ValueError(
            f"Data quality check failed: Total amount of null values in {table_name} table is {total_nulls}"
        )

# Validate NULL Values in the Fact Table

This step verifies that the key columns in the `fact_weather` table do not contain NULL values.

The validation covers the following columns:

- `city_id` – foreign key referencing the `dim_city` table.
- `date_key` – foreign key referencing the `dim_date` table.
- `measurement_time` – timestamp identifying the hourly weather measurement.

These columns are essential for identifying each weather record and maintaining relationships with the dimension tables.

The pipeline stops if NULL values are detected in any of these columns.

In [0]:
# Create list of columns in fact_weather that must not be null
fact_weather_critical_columns = ["city_id", "date_key", "measurement_time"]

# Use tables_null_validation function to check for nulls
tables_null_validation(fact_weather_df, "fact_weather", *fact_weather_critical_columns)

# Validate NULL Values in the City Dimension

This step verifies that the `dim_city` table does not contain NULL values in its key and descriptive columns.

The validation covers the following columns:

- `city_id` – surrogate key uniquely identifying each city.
- `city` – city name.
- `latitude` – geographic latitude of the city.
- `longitude` – geographic longitude of the city.

All four columns are required to ensure that each city is uniquely identified and has complete geographic information.

The pipeline stops if NULL values are detected in any of these columns.

In [0]:
# Create a list of columns in dim_city to check for null values
dim_city_columns = ["city_id", "city", "latitude", "longitude"]

# Use tables_null_validation function to check for nulls
tables_null_validation(dim_city, "dim_city", *dim_city_columns)

# Validate NULL Values in the Date Dimension

This step verifies that the `dim_date` table does not contain NULL values in its required calendar attributes.

The validation covers the following columns:

- `date_key` – surrogate key uniquely identifying each calendar date.
- `full_date` – full calendar date.
- `day` – day of the month.
- `month` – calendar month number.
- `month_name` – calendar month name.
- `quarter` – calendar quarter.
- `year` – calendar year.
- `day_of_week` – day of the week.
- `is_weekend` – indicator identifying weekends.

All columns are required to ensure that the date dimension contains complete and reliable calendar information for analytical reporting.

The pipeline stops if NULL values are detected in any of these columns.

In [0]:
# Create a list of columns in dim_date to check for null values
dim_date_columns = [
    "date_key",
    "full_date",
    "day",
    "month",
    "month_name",
    "quarter",
    "year",
    "day_of_week",
    "is_weekend",
]

# Use tables_null_validation function to check for nulls
tables_null_validation(dim_date, "dim_date", *dim_date_columns)